# El meta-modelo con otras semillas

Este notebook es el paso 3 de la carpeta `semillas`. Para cada semilla nueva junta el XGBoost (A) y la red (B) **de esa misma semilla**, entrena el meta-modelo con el código de `meta_modelo/1_entrenamiento.ipynb` y elige su umbral de confianza con las reglas de `meta_modelo/2_confianza.ipynb`. Así, cada semilla es un sistema completo, entrenado de punta a punta con ella.

En el meta-modelo, la semilla fija la regresión logística y el submuestreo de filas y columnas de su XGBoost (`random_state`). Las entradas, los hiperparámetros y los folds no cambian.

Necesita las predicciones de `1_entrenamiento_red.ipynb` y de `2_entrenamiento_xgboost.ipynb`.

El desarrollo se organiza de la siguiente manera:

1. **Librerías**
2. **Rutas y parámetros**
3. **Predicciones de A y B**
4. **Entradas del meta-modelo**
5. **Entrenamiento**
6. **Umbral de confianza**
7. **Modelos guardados**
8. **Conclusiones**

## 1. Librerías

In [1]:
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 2. Rutas y parámetros

El meta-modelo de cada semilla se guarda en `experimentacion/semillas/modelos/semilla_S/meta_modelo/`, junto a su `sistema.json`, y sus predicciones en `data/procesados/semillas/semilla_S/meta_modelo/`.

In [2]:
# Raíz del repositorio: la carpeta que tiene particiones.json
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "particiones.json").exists())
SALIDA = RAIZ / "data" / "procesados" / "semillas"
MODELOS = RAIZ / "experimentacion" / "semillas" / "modelos"
SISTEMA_42 = RAIZ / "modelo" / "sistema.json"                                   # el sistema en producción
META_42 = RAIZ / "data" / "procesados" / "meta_modelo" / "predicciones.npz"

SEMILLAS = [1, 2, 3, 4]
ESTADIOS = ["W", "N1", "N2", "N3", "REM"]
K = len(ESTADIOS)
MODALIDADES = ("fpz", "pz", "eog", "emg")

# Reglas de confianza de 2_confianza.ipynb
UMBRALES = np.round(np.arange(0.50, 0.951, 0.05), 2)
ACIERTO_MINIMO = 0.90
PERCENTIL_AVISO = 0.90

particion = json.loads((RAIZ / "particiones.json").read_text())
CONFIGS = {k: tuple(v) for k, v in particion["combinaciones"].items()}
FOLDS = [f"fold_{f['fold']}" for f in particion["folds"]]


def mascara(cfg):
    """[fpz, pz, eog, emg] -> True si el canal está en la combinación."""
    return [m in CONFIGS[cfg] for m in MODALIDADES]


def carpeta(semilla, pieza):
    """Lo que dejó cada pieza de una semilla: modelo_a_xgboost_100_features, modelo_b_red o meta_modelo."""
    return SALIDA / f"semilla_{semilla}" / pieza

## 3. Predicciones de A y B

Como en `meta_modelo/1_entrenamiento.ipynb`: las de los 5 folds juntas dan las predicciones fuera de fold de los 62 sujetos de desarrollo, y las del modelo final, las del test. B solo predice las épocas con etiqueta, así que sus predicciones se ubican en la ventana de A y las épocas sin etiqueta toman la de la época anterior.

In [3]:
# Función para ubicar las predicciones de B en la ventana de A
def fill_window(y_window, y_valid, p_valid):
    valid = y_window >= 0
    assert np.array_equal(y_window[valid], y_valid), "las etiquetas de A y B no coinciden"
    idx = np.where(valid, np.arange(len(y_window)), -1)
    idx = np.maximum.accumulate(idx)
    idx[idx < 0] = np.argmax(valid)
    full = np.empty((len(y_window), p_valid.shape[1]), dtype=np.float32)
    full[valid] = p_valid
    full[~valid] = full[idx[~valid]]
    return full

In [4]:
# Funciones para leer las predicciones de A y de B de una semilla
def load_a(semilla, name):
    """combinación -> noche -> (y, probabilidades)"""
    z = np.load(carpeta(semilla, "modelo_a_xgboost_100_features") / "folds" / f"{name}.npz")
    out = {}
    for k in z.files:
        cfg, n, kind = k.split("__")
        if kind == "y":
            out.setdefault(cfg, {})[n] = (z[k], z[f"{cfg}__{n}__pred"])
    return out


def load_b(semilla, name, key, A):
    """Como load_a, con las predicciones de B ubicadas en la ventana de A."""
    z = np.load(carpeta(semilla, "modelo_b_red") / "predicciones" / f"{name}.npz")
    out = {c: {} for c in CONFIGS}
    for k in z.files:
        cfg, n = k.split("__")
        if cfg in CONFIGS:
            yw = A[cfg][key][n][0]
            out[cfg][n] = (yw, fill_window(yw, z[f"y__{n}"], z[k]))
    return out


def predicciones_ab(semilla):
    """A y B de una semilla: combinación -> 'oof'/'test' -> noche -> (y, p)"""
    A = {c: {"oof": {}, "test": {}} for c in CONFIGS}
    B = {c: {"oof": {}, "test": {}} for c in CONFIGS}
    for name in FOLDS + ["final"]:
        key = "test" if name == "final" else "oof"
        for c, d in load_a(semilla, name).items():
            A[c][key].update(d)
    for name in FOLDS + ["final"]:
        key = "test" if name == "final" else "oof"
        for c, d in load_b(semilla, name, key, A).items():
            B[c][key].update(d)
    for c in CONFIGS:
        for key in ("oof", "test"):
            assert sorted(B[c][key]) == sorted(A[c][key]), f"{c}: A y B no tienen las mismas noches"
    return A, B

In [5]:
# Ejemplo: las predicciones de la semilla 1
A1, B1 = predicciones_ab(SEMILLAS[0])
n = sorted(A1["fpz"]["oof"])[0]
y = A1["fpz"]["oof"][n][0]
print(f"semilla {SEMILLAS[0]}: {len(A1['fpz']['oof'])} noches de CV y {len(A1['fpz']['test'])} de test")
print(f"{n}: {len(y)} épocas en la ventana, {(y < 0).sum()} sin etiqueta · A y B alineados: "
      f"{A1['fpz']['oof'][n][1].shape} y {B1['fpz']['oof'][n][1].shape}")

semilla 1: 121 noches de CV y 32 de test
SC4011: 1103 épocas en la ventana, 0 sin etiqueta · A y B alineados: (1103, 5) y (1103, 5)


## 4. Entradas del meta-modelo

Las mismas 54 entradas: las log-probabilidades de A y de B en las épocas t−2 a t+2 (50) y la máscara de los 4 canales.

In [6]:
# Función para armar las 54 entradas de una noche
EPS = 1e-6
CONTEXT = (-2, -1, 0, 1, 2)


def entradas(pa, pb, mask):
    """pa, pb: (épocas, 5) probabilidades de A y B; mask: [fpz, pz, eog, emg]."""
    cols = {}
    for nombre, prob in (("xgb", pa), ("b", pb)):
        lp = np.log(np.clip(prob, EPS, 1))
        for o in CONTEXT:
            shifted = np.full_like(lp, np.log(1 / K))
            if o == 0:
                shifted = lp
            elif o > 0:
                shifted[:-o] = lp[o:]
            else:
                shifted[-o:] = lp[:o]
            for k in range(K):
                cols[f"{nombre}_k{k}_o{o:+d}"] = shifted[:, k]
    df = pd.DataFrame(cols).astype(np.float32)
    for m, on in zip(MODALIDADES, mask):
        df[f"mask_{m}"] = np.float32(on)
    return df

In [7]:
# Entradas de todas las combinaciones y noches
def tabla(A, B, key):
    frames = []
    for c in CONFIGS:
        for n in sorted(A[c][key]):
            y, pa = A[c][key][n]
            d = entradas(pa, B[c][key][n][1], mascara(c))
            d.insert(0, "y", y)
            d.insert(0, "night", n)
            d.insert(0, "config", c)
            frames.append(d)
    return pd.concat(frames, ignore_index=True)


oof_ej = tabla(A1, B1, "oof")
feat = [c for c in oof_ej.columns if c not in ("config", "night", "y")]
print(f"semilla {SEMILLAS[0]} · CV: {len(oof_ej):,} filas · {len(feat)} entradas")
del oof_ej

semilla 1 · CV: 1,889,520 filas · 54 entradas


## 5. Entrenamiento

### Meta-modelo

Regresión logística + XGBoost pequeño, promediados, con los mismos hiperparámetros y pesos por clase. La clase ahora recibe la semilla.

In [8]:
# Hiperparámetros del meta-modelo y pesos por clase
ALPHA = 0.44
XGB_PARAMS = dict(objective="multi:softprob", tree_method="hist", eval_metric="mlogloss", learning_rate=0.05,
                  max_depth=4, subsample=0.8, colsample_bytree=0.8, min_child_weight=10, n_jobs=-1)


def class_weights(y, alpha=ALPHA):
    return compute_sample_weight("balanced", y) ** alpha

In [9]:
# El meta-modelo: regresión logística + XGBoost, promediados, con una semilla
class MetaModelo:
    def __init__(self, semilla):
        self.semilla = semilla

    def fit(self, X, y, X_val, y_val):
        lr = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000, random_state=self.semilla))
        Xt, yt = pd.concat([X, X_val]), np.concatenate([y, y_val])   # la LR no necesita validación: usa todo
        lr.fit(Xt, yt, logisticregression__sample_weight=class_weights(yt))
        m = xgb.XGBClassifier(**XGB_PARAMS, n_estimators=1000, early_stopping_rounds=50, random_state=self.semilla)
        m.fit(X, y, sample_weight=class_weights(y), eval_set=[(X_val, y_val)],
              sample_weight_eval_set=[class_weights(y_val)], verbose=False)
        self.lr, self.xgb = lr, m
        return self

    def predict_proba(self, X):
        return np.mean([self.lr.predict_proba(X), self.xgb.predict_proba(X)], axis=0)

### Las 4 semillas

Para cada semilla: validación cruzada con los 5 folds (las predicciones fuera de fold de la CV) y el modelo final con el split `final`, que predice el test. Si una semilla ya está entrenada, se salta.

In [10]:
# Función para entrenar el meta-modelo de una semilla y guardarlo
def per_config(df, proba):
    """probabilidades de la tabla -> combinación -> noche -> (y, p)"""
    out = {}
    for (c, n), i in df.groupby(["config", "night"]).indices.items():
        out.setdefault(c, {})[n] = (df.y.values[i], proba[i].astype(np.float32))
    return out


def entrenar_semilla(s):
    """CV con los 5 folds y modelo final; guarda el modelo y las predicciones. Devuelve las rondas del XGBoost."""
    salida = carpeta(s, "meta_modelo")
    if (salida / "rondas.json").exists():   # ya entrenado
        return json.loads((salida / "rondas.json").read_text())
    A, B = predicciones_ab(s)
    oof_in, test_in = tabla(A, B, "oof"), tabla(A, B, "test")
    sujeto = oof_in.night.str[3:5].astype(int)
    con_etiqueta = oof_in.y.values >= 0

    def fit_meta(train_subj, val_subj):
        tr = con_etiqueta & sujeto.isin(train_subj).values
        va = con_etiqueta & sujeto.isin(val_subj).values
        return MetaModelo(s).fit(oof_in.loc[tr, feat], oof_in.y.values[tr], oof_in.loc[va, feat], oof_in.y.values[va])

    rondas = {}
    p_oof = np.zeros((len(oof_in), K), dtype=np.float32)
    for f in particion["folds"]:
        m = fit_meta(f["train"], f["val"])
        r = sujeto.isin(f["test"]).values
        p_oof[r] = m.predict_proba(oof_in.loc[r, feat])
        rondas[f"fold {f['fold'] + 1}"] = m.xgb.best_iteration + 1
    meta_final = fit_meta(particion["final"]["train"], particion["final"]["val"])
    rondas["final"] = meta_final.xgb.best_iteration + 1
    p_test = meta_final.predict_proba(test_in[feat]).astype(np.float32)

    destino = MODELOS / f"semilla_{s}" / "meta_modelo"
    destino.mkdir(parents=True, exist_ok=True)
    joblib.dump(meta_final.lr, destino / "meta_lr.joblib")
    meta_final.xgb.save_model(destino / "meta_xgb.json")
    (destino / "columnas.json").write_text(json.dumps(feat))

    salida.mkdir(parents=True, exist_ok=True)
    arr = {}
    for key, P in (("oof", per_config(oof_in, p_oof)), ("test", per_config(test_in, p_test))):
        for c, d in P.items():
            for n, (y, p) in d.items():
                arr[f"{key}__{c}__{n}__y"], arr[f"{key}__{c}__{n}__pred"] = y, p
    np.savez_compressed(salida / "predicciones.npz", **arr)
    (salida / "rondas.json").write_text(json.dumps(rondas))
    return rondas

In [ ]:
# Entrenamiento de las 4 semillas
RONDAS = {}
for s in SEMILLAS:
    RONDAS[s] = entrenar_semilla(s)
    print(f"semilla {s}: XGBoost del meta con {RONDAS[s]['final']} rondas · guardado en "
          f"{(MODELOS / f'semilla_{s}' / 'meta_modelo').relative_to(RAIZ)}")

semilla 1: XGBoost del meta con 256 rondas · guardado en experimentacion/semillas/modelos/semilla_1/meta_modelo
semilla 2: XGBoost del meta con 238 rondas · guardado en experimentacion/semillas/modelos/semilla_2/meta_modelo
semilla 3: XGBoost del meta con 183 rondas · guardado en experimentacion/semillas/modelos/semilla_3/meta_modelo
semilla 4: XGBoost del meta con 285 rondas · guardado en experimentacion/semillas/modelos/semilla_4/meta_modelo


Rondas que eligió el early stopping del XGBoost del meta-modelo:

In [12]:
# Rondas del XGBoost del meta-modelo en cada fold y en el final
pd.DataFrame({f"semilla {s}": r for s, r in RONDAS.items()}).T

,fold 1,fold 2,fold 3,fold 4,fold 5,final
semilla 1,299,212,155,243,148,256
semilla 2,460,208,203,281,164,238
semilla 3,367,219,166,318,151,183
semilla 4,362,228,183,274,167,285


MF1 de la CV del meta-modelo, junto al de la semilla 42 (`meta_modelo/1_entrenamiento.ipynb`). El test se compara en `4_resumen.ipynb`.

In [13]:
# Función para leer las predicciones del meta-modelo de una semilla
def load_meta(semilla):
    """'oof'/'test' -> combinación -> noche -> (y, probabilidades)"""
    z = np.load(META_42 if semilla == 42 else carpeta(semilla, "meta_modelo") / "predicciones.npz")
    out = {"oof": {}, "test": {}}
    for k in z.files:
        key, c, n, kind = k.split("__")
        if kind == "y":
            out[key].setdefault(c, {})[n] = (z[k], z[f"{key}__{c}__{n}__pred"])
    return out


META = {s: load_meta(s) for s in [42, *SEMILLAS]}

In [14]:
# MF1 de la CV del meta-modelo por combinación y semilla
def metrics_of(preds):
    y = np.concatenate([y for y, _ in preds.values()])
    p = np.concatenate([p.argmax(1) for _, p in preds.values()])
    v = y >= 0
    return {"mf1": f1_score(y[v], p[v], labels=range(K), average="macro", zero_division=0)}


cv = pd.DataFrame({f"semilla {s}": {c: metrics_of(META[s]["oof"][c])["mf1"] for c in CONFIGS} for s in META})
cv.assign(media=cv.mean(1), de=cv.std(1)).style.format(precision=3)

,semilla 42,semilla 1,semilla 2,semilla 3,semilla 4,media,de
fpz,0.799,0.802,0.803,0.800,0.799,0.801,0.002
pz,0.760,0.762,0.761,0.761,0.762,0.761,0.001
eeg2,0.809,0.811,0.811,0.810,0.810,0.810,0.001
fpz_eog,0.808,0.811,0.812,0.809,0.810,0.810,0.001
pz_eog,0.788,0.788,0.790,0.788,0.788,0.788,0.001
eeg2_eog,0.817,0.817,0.817,0.817,0.817,0.817,0.000
fpz_emg,0.802,0.803,0.804,0.801,0.801,0.802,0.001
pz_emg,0.764,0.766,0.765,0.763,0.765,0.765,0.001
eeg2_emg,0.810,0.813,0.812,0.811,0.811,0.811,0.001
fpz_eog_emg,0.810,0.812,0.812,0.811,0.811,0.811,0.001


## 6. Umbral de confianza

Con las reglas fijas de `meta_modelo/2_confianza.ipynb`, aplicadas a la CV de cada semilla:
- **umbral:** el menor de 0.50, 0.55, …, 0.95 con el que lo que el sistema entrega solo acierta al menos el 90 %;
- **aviso por noche:** el percentil 90 de la fracción de cada noche que queda para revisar.

In [15]:
# Funciones para resumir una noche con un umbral
def _f1_macro(cm):
    tp, rows, cols = np.diag(cm), cm.sum(1), cm.sum(0)
    return np.divide(2 * tp, rows + cols, out=np.zeros(K), where=(rows + cols) > 0).mean()


def night_stats(y, p, umbral):
    v = y >= 0
    y, p = y[v], p[v]
    o = np.argsort(-p, 1)
    conf, t1, t2 = p.max(1), o[:, 0], o[:, 1]
    seg, ok, top2 = conf >= umbral, t1 == y, (t1 == y) | (t2 == y)
    cm = np.bincount(y[seg] * K + t1[seg], minlength=K * K).reshape(K, K)
    return {"n": len(y), "seg": seg.sum(), "ok_seg": (ok & seg).sum(), "dud": (~seg).sum(), "ok_dud": (ok & ~seg).sum(),
            "top2_dud": (top2 & ~seg).sum(), "ok": ok.sum(), "combinado": (ok & seg).sum() + (top2 & ~seg).sum(), "cm_seg": cm}


def summarize(stats):
    t = {k: sum(s[k] for s in stats) for k in stats[0]}
    return {"entrega": t["seg"] / t["n"], "acierto ahí": t["ok_seg"] / t["seg"], "MF1 ahí": _f1_macro(t["cm_seg"]),
            "acierto en revisar": t["ok_dud"] / max(t["dud"], 1), "2 opciones en revisar": t["top2_dud"] / max(t["dud"], 1),
            "acierto sin revisión": t["ok"] / t["n"], "acierto con revisión": t["combinado"] / t["n"]}

In [16]:
# Umbral y aviso de cada semilla, elegidos con su CV
def umbral_y_aviso(meta_oof):
    noches_cv = [(c, n) for c in CONFIGS for n in meta_oof[c]]
    curva = pd.DataFrame([{"umbral": u, **summarize([night_stats(*meta_oof[c][n], u) for c, n in noches_cv])}
                          for u in UMBRALES]).set_index("umbral")
    umbral = float(curva.index[curva["acierto ahí"] >= ACIERTO_MINIMO][0])
    frac = np.array([s["dud"] / s["n"] for s in (night_stats(*meta_oof[c][n], umbral) for c, n in noches_cv)])
    return umbral, float(np.quantile(frac, PERCENTIL_AVISO)), curva.loc[umbral]


filas = []
REGLAS = {}
for s in SEMILLAS:
    umbral, aviso, fila = umbral_y_aviso(META[s]["oof"])
    REGLAS[s] = (umbral, aviso)
    filas.append({"semilla": s, "umbral": umbral, "aviso": round(aviso, 4), "entrega (CV)": fila["entrega"],
                  "acierto ahí (CV)": fila["acierto ahí"]})
sistema_42 = json.loads(SISTEMA_42.read_text())
filas.insert(0, {"semilla": 42, "umbral": sistema_42["umbral_confianza"], "aviso": sistema_42["aviso_noche_fraccion_revisar"]})
pd.DataFrame(filas).set_index("semilla").style.format(precision=3, na_rep="")

,umbral,aviso,entrega (CV),acierto ahí (CV)
semilla,,,,
42,0.650,0.277,,
1,0.650,0.267,0.826,0.910
2,0.650,0.263,0.828,0.909
3,0.650,0.272,0.824,0.909
4,0.650,0.264,0.827,0.909


Cada semilla guarda su `sistema.json`: el mismo del sistema en producción, con su umbral, su aviso y la semilla.

In [17]:
# sistema.json de cada semilla: el del sistema en producción, con su umbral y su aviso
for s in SEMILLAS:
    umbral, aviso = REGLAS[s]
    sistema = {**sistema_42, "umbral_confianza": umbral, "aviso_noche_fraccion_revisar": round(aviso, 4), "semilla": s}
    ruta = MODELOS / f"semilla_{s}" / "sistema.json"
    ruta.write_text(json.dumps(sistema, indent=2, ensure_ascii=False) + "\n")
    print("guardado:", ruta.relative_to(RAIZ))

guardado: experimentacion/semillas/modelos/semilla_1/sistema.json
guardado: experimentacion/semillas/modelos/semilla_2/sistema.json
guardado: experimentacion/semillas/modelos/semilla_3/sistema.json
guardado: experimentacion/semillas/modelos/semilla_4/sistema.json


## 7. Modelos guardados

Cada semilla queda con las mismas piezas que `modelo/`: el XGBoost, la red, el meta-modelo y `sistema.json`.

In [18]:
# Las piezas de cada semilla, con las mismas carpetas que modelo/
for s in SEMILLAS:
    d = MODELOS / f"semilla_{s}"
    archivos = sorted(str(p.relative_to(d)) for p in d.rglob("*") if p.is_file())
    print(f"semilla {s}: " + " · ".join(archivos))

semilla 1: meta_modelo/columnas.json · meta_modelo/meta_lr.joblib · meta_modelo/meta_xgb.json · modelo_a_xgboost_100_features/columnas.json · modelo_a_xgboost_100_features/xgb.json · modelo_b_red/modelo_final.pt · sistema.json
semilla 2: meta_modelo/columnas.json · meta_modelo/meta_lr.joblib · meta_modelo/meta_xgb.json · modelo_a_xgboost_100_features/columnas.json · modelo_a_xgboost_100_features/xgb.json · modelo_b_red/modelo_final.pt · sistema.json
semilla 3: meta_modelo/columnas.json · meta_modelo/meta_lr.joblib · meta_modelo/meta_xgb.json · modelo_a_xgboost_100_features/columnas.json · modelo_a_xgboost_100_features/xgb.json · modelo_b_red/modelo_final.pt · sistema.json
semilla 4: meta_modelo/columnas.json · meta_modelo/meta_lr.joblib · meta_modelo/meta_xgb.json · modelo_a_xgboost_100_features/columnas.json · modelo_a_xgboost_100_features/xgb.json · modelo_b_red/modelo_final.pt · sistema.json


## 8. Conclusiones

- El meta-modelo también es estable: en las 12 combinaciones, su MF1 de la CV varía entre semillas con una desviación estándar de 0.002 o menos, aunque el early stopping elija rondas distintas (el final va de 183 a 294).
- La regla de confianza elige el mismo umbral, **0.65**, en las 5 semillas, y el aviso por noche queda entre 26.3 % y 27.7 %.
- Cada semilla queda como un sistema completo, con las mismas piezas que `modelo/`. El test de las 5 se compara en `4_resumen.ipynb`.